# LAB 05 — Linear Regression: Predicting House Prices

## Student Information
**Name:**
**Student ID:**
**Section:**
**GitHub:**
**Kaggle:**
**Dataset:** House Prices — Advanced Regression Techniques
**Dataset Source:** Kaggle — https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques

**Duration:** 3 Hours (independent, hands-on)

> This notebook is a **template**, not a tutorial. You already know Python, Pandas, and Linear Regression theory. Each section states the objective and the questions you must answer — **you write the code**. Every table, plot, and metric needs a short written observation.
>
> This dataset has 79 raw features and comes from a real, ongoing Kaggle competition. Full feature engineering across all 79 columns is out of scope for this lab — select a manageable subset of roughly 10–12 features (mixing numerical and categorical) that you'd reasonably expect to matter for price.
>
> Submit **only this one notebook**. Keep every Markdown heading below — they are used for grading. Fill in the empty code cells (`# YOUR CODE HERE`) and the *Answer:* placeholders directly in this notebook.

---

## Setup

Import what you'll need. Add any other scikit-learn imports as you go.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


---

## 1. Problem Definition

Before writing any modeling code, understand what you're predicting and what you're predicting it from.

**Tasks**
1. In 1–2 sentences, describe the business/practical problem: what does this dataset let you predict, and why would that be useful?
2. In 1–2 sentences, state why this is a regression problem rather than a classification problem, and what that changes about how you'll evaluate the model later.

**Answer 1:**
This dataset helps us predict the final selling price of a house using its features, such as its size, location, age, and overall condition. Being able to estimate house prices accurately helps buyers and sellers negotiate fair deals, allows banks to evaluate home loan amounts safely, and helps real estate apps give automated price estimates.

**Answer 2:**
This is a regression problem because the target we are predicting (SalePrice) is a continuous number (money in dollars), not a category like "cheap vs. expensive" or "yes vs. no." Because of this, we cannot evaluate the model using classification accuracy or confusion matrices; instead, we must measure how far off our numerical guesses are using regression metrics like Mean Squared Error (MSE) or Root Mean Squared Logarithmic Error (RMSLE).

---

## 2. Dataset Exploration

*Concepts/functions you may find useful:* `pd.read_csv()`, `df.shape`, `df.info()`, `df.head()`, `df.describe()`

**Task**
3. Load `train.csv` and briefly inspect it (shape, dtypes, first few rows, summary statistics). This dataset has 79 feature columns — you don't need to explore all of them in depth; skim to get a general sense, then focus on Section 3.

In [4]:
# Task 3 — load and inspect the dataset
df = pd.read_csv("train.csv")
print("shape",df.shape)
print("information",df.info())
print("statistics",df.describe())
print(df.head())

shape (1460, 81)
<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  
 18  Ov

**Observations:**

* **Dataset Size & Shape:** The dataset contains 1,460 rows and 81 columns (38 numerical, 43 categorical). The high number of features relative to the sample size means encoding will expand the feature space significantly, making regularized regression (Ridge or Lasso) necessary to avoid overfitting.
* **Target Variable (`SalePrice`) Skewness:** The target is right-skewed, with the mean ($180,921) noticeably higher than the median ($163,000). Prices range from $34,900 to $755,000, with a steep jump between the 75th percentile ($214,000) and the maximum. A log transformation ($\log(1+y)$) is required to stabilize variance and meet normality assumptions.
* **Structural Missingness (`NaN` as "None"):** Features like `PoolQC` (only 7 non-null), `MiscFeature` (54 non-null), `Alley` (91 non-null), and `Fence` (281 non-null) have extremely low fill rates. These are not missing at random; the data description states that `NaN` indicates the absence of that amenity (e.g., no pool, no alley access), so they should be imputed as `'None'`.
* **Zero-Inflated Numerical Features:** Several numeric columns—including `WoodDeckSF`, `OpenPorchSF`, `EnclosedPorch`, `3SsnPorch`, `ScreenPorch`, `PoolArea`, and `MiscVal`—have a median and 25th percentile of 0. Most houses simply do not have these additions.
* **Scale Discrepancies & Outliers:** Feature scales vary by orders of magnitude—from small integer ratings like `OverallQual` (1–10) to `LotArea` (ranging up to 215,245 sq ft). This large discrepancy makes feature scaling (`RobustScaler` or `StandardScaler`) essential prior to fitting regularized linear models.

---

## 3. Feature and Target Identification

**Task**
4. Select a manageable subset of roughly 10–12 features from the 79 available (mixing numerical and categorical) that you'd reasonably expect to matter for price. Identify the dependent variable, and list your chosen independent variables split explicitly into numerical features and categorical features.
new_df = df[[
    # Dependent variable (Target)
    "SalePrice",
    
    # Numerical independent variables
    "GrLivArea",
    "TotalBsmtSF",
    "OverallQual",
    "YearBuilt",
    "GarageCars",
    "FullBath",
    "LotArea",
    
    # Categorical independent variables
    "Neighborhood",
    "ExterQual",
    "KitchenQual",
    "CentralAir",
    "BldgType"
]]

**Answer 4:**

**Dependent variable:**
* `SalePrice` (the sale price of the house in dollars)

**Numerical independent variables:**
1. `GrLivArea` (above grade living area in square feet)
2. `TotalBsmtSF` (total square feet of basement area)
3. `OverallQual` (overall material and finish quality, rated 1–10)
4. `YearBuilt` (original construction year)
5. `GarageCars` (size of garage in car capacity)
6. `FullBath` (number of full bathrooms above grade)
7. `LotArea` (lot size in square feet)

**Categorical independent variables:**
1. `Neighborhood` (physical location within Ames city limits)
2. `ExterQual` (evaluates the quality of exterior material: Ex, Gd, TA, Fa)
3. `KitchenQual` (kitchen quality: Ex, Gd, TA, Fa)
4. `CentralAir` (central air conditioning: Y, N)
5. `BldgType` (type of dwelling: 1Fam, 2FmCon, Duplx, TwnhsE, TwnhsI)

---

## 4. Data Preparation

scikit-learn's `LinearRegression` requires fully numeric input. Convert categorical columns appropriately.

*Concepts/functions you may find useful:* `df[col].map({'yes': 1, 'no': 0})`, `pd.get_dummies(df, columns=[...])`

**Tasks**
5. Prepare `X` (features) and `y` (target) for modeling: encode every categorical column into numeric form.
6. Briefly justify your choice of encoding method for each type of column.
7. Confirm `X` is fully numeric with no missing values.

In [5]:
# Task 5 — define X and y, encode categorical columns
# Dependent variable (target)
y = df["SalePrice"]

# Independent variables (all 12 selected features)
X = df[[
    "GrLivArea",
    "TotalBsmtSF",
    "OverallQual",
    "YearBuilt",
    "GarageCars",
    "FullBath",
    "LotArea",
    "Neighborhood",
    "ExterQual",
    "KitchenQual",
    "CentralAir",
    "BldgType"
]]
X = pd.get_dummies(X, drop_first=True, dtype=int)

In [6]:
# Task 7 — confirm X is fully numeric with no missing values
X.head()

,GrLivArea,TotalBsmtSF,OverallQual,YearBuilt,GarageCars,FullBath,LotArea,Neighborhood_Blueste,Neighborhood_BrDale,Neighborhood_BrkSide,...,ExterQual_Gd,ExterQual_TA,KitchenQual_Fa,KitchenQual_Gd,KitchenQual_TA,CentralAir_Y,BldgType_2fmCon,BldgType_Duplex,BldgType_Twnhs,BldgType_TwnhsE
0,1710,856,7,2003,2,2,8450,0,0,0,...,1,0,0,1,0,1,0,0,0,0
1,1262,1262,6,1976,2,2,9600,0,0,0,...,0,1,0,0,1,1,0,0,0,0
2,1786,920,7,2001,2,2,11250,0,0,0,...,1,0,0,1,0,1,0,0,0,0
3,1717,756,7,1915,3,1,9550,0,0,0,...,0,1,0,1,0,1,0,0,0,0
4,2198,1145,8,2000,3,2,14260,0,0,0,...,1,0,0,1,0,1,0,0,0,0


**Answer 6 (encoding justification):**
_(your answer here)_

---

## 5. Train/Test Split

*Concepts/functions you may find useful:* `train_test_split(X, y, test_size=..., random_state=...)`

**Task**
8. Split `X` and `y` into training and test sets (e.g. an 80/20 split) and report the resulting shapes.

In [9]:
# Task 8 — train/test split, report shapes
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
print("training shape of x",X_train.shape)
print("training shape of y",y_train.shape)
print("testing shape of x",X_test.shape)
print("testing shape of y",y_test.shape)

training shape of x (1168, 42)
training shape of y (1168,)
testing shape of x (292, 42)
testing shape of y (292,)


---

## 6. Linear Regression Model

*Concepts/functions you may find useful:* `LinearRegression()`, `.fit(X_train, y_train)`

**Task**
9. Train a `LinearRegression` model on the training data.

In [11]:
# Task 9 — train the Linear Regression model
lr_model = LinearRegression()
lr_model.fit(X_train,y_train)


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


---

## 7. Predictions

*Concepts/functions you may find useful:* `.predict(X)`

**Task**
10. Generate predicted prices on the test set, and display a small table comparing actual vs. predicted price for 10 test houses.

In [13]:
# Task 10 — generate predictions, compare actual vs predicted
y_pred = lr_model.predict(X_test)
comparison_df = pd.DataFrame({
    'Actual Price': y_test.values[:10],
    'Predicted Price': np.round(y_pred[:10], 2),
    'Difference (Error)': np.round(y_pred[:10] - y_test.values[:10], 2),
    'Absolute % Error': np.round(np.abs(y_test.values[:10] - y_pred[:10]) / y_test.values[:10] * 100, 2)
})
comparison_df.head()


,Actual Price,Predicted Price,Difference (Error),Absolute % Error
0,154500,136659.79,-17840.21,11.55
1,325000,320935.59,-4064.41,1.25
2,115000,115185.13,185.13,0.16
3,159000,173058.84,14058.84,8.84
4,315500,295785.45,-19714.55,6.25


**Task 10 — Actual vs. Predicted (sample of 10)**

| Actual Price | Predicted Price |
|---|---|
| | |
| | |
| | |

---

## 8. Coefficients and Intercept

*Concepts/functions you may find useful:* `model.intercept_`, `model.coef_`, `pd.DataFrame({'Feature': X.columns, 'Coefficient': model.coef_})`

**Tasks**
11. Extract the intercept and all coefficients. Build a table: Feature | Coefficient | Interpretation, with a one-line, plain-English interpretation for each feature.
12. Does the intercept have a sensible real-world interpretation for this dataset? Why or why not?

In [ ]:
# Task 11 — extract intercept and coefficients


**Task 11 — Coefficient Table**

| Feature | Coefficient | Interpretation |
|---|---|---|
| | | |
| | | |
| | | |

**Answer 12:**
_(your answer here)_

---

## 9. Residual Analysis

*Concepts/functions you may find useful:* `residuals = y_test - y_pred`, `plt.scatter(y_pred, residuals)`, `plt.hist(residuals, bins=...)`

**Task**
13. Compute residuals on the test set and plot them (e.g. residuals vs. predicted values, and a histogram of residuals). Do the residuals look randomly scattered, or is there a visible pattern?

In [ ]:
# Task 13 — compute and plot residuals


**Answer 13:**
_(your answer here)_

---

## 10. Regression Metrics

*Concepts/functions you may find useful:* `mean_absolute_error`, `mean_squared_error`, `r2_score` (RMSE = sqrt of MSE)

**Tasks**
14. Compute MAE, MSE, RMSE, and R² on the test set. Report them in a small table.
15. In your own words, what does each metric tell you, and which one would you show to a non-technical stakeholder? Why?

In [ ]:
# Task 14 — compute MAE, MSE, RMSE, R² on the test set


**Task 14 — Test Set Metrics**

| Metric | Value |
|---|---|
| MAE | |
| MSE | |
| RMSE | |
| R² | |

**Answer 15:**
_(your answer here)_

---

## 11. Training vs Testing Performance

*Concepts/functions you may find useful:* `model.predict(X_train)`, comparing metrics computed on train vs. test

**Tasks**
16. Compute MAE, MSE, RMSE, and R² on the training set, and place them side by side with your Section 10 test-set metrics in one comparison table.
17. Is there a large gap between training and testing performance? What would a large gap suggest about overfitting or generalization, and what does your actual result suggest here?

In [ ]:
# Task 16 — compute training-set metrics


**Task 16 — Train vs. Test Comparison**

| Metric | Training Set | Test Set |
|---|---|---|
| MAE | | |
| MSE | | |
| RMSE | | |
| R² | | |

**Answer 17:**
_(your answer here)_

---

## 12. Model Interpretation

**Task**
18. Looking back at your Section 8 coefficient table, which 2–3 features appear most influential on price? Justify your answer using the coefficient magnitudes (careful: a larger coefficient does not automatically mean a more important feature if the underlying scales differ — discuss this briefly).

**Answer 18:**
_(your answer here)_

---

## 13. Conclusion

**Tasks**
19. Write a short final interpretation (5–7 sentences) answering: How well does the model predict house prices? Which metric is most useful and why? Which feature has the strongest effect? What limitations does Linear Regression have for this dataset? What would you try next?
20. Apply the same feature selection and encoding you used on `train.csv` to `test.csv`, predict `SalePrice` for every row, and build a submission file with exactly two columns: `Id` and `SalePrice` (match `sample_submission.csv`'s format). Save it as `submission.csv`.
21. Submit `submission.csv` on the competition's "Submit Predictions" page. Record your public leaderboard score and rank below.

**Answer 19 — Conclusion:**
_(your answer here)_

In [ ]:
# Task 20 — apply encoding to test.csv, predict SalePrice, build submission.csv


**Answer 21 — Kaggle Leaderboard Result:**

Public score: _______

Rank: _______

Reflection (1–2 sentences): _(your answer here)_

---

## Submission Checklist

Before submitting, confirm you have:

- [ ] Filled in the student-info block at the top (Name, Student ID, Section, GitHub, Kaggle, Dataset, Dataset Source)
- [ ] Section 1 — Problem Definition
- [ ] Section 2 — Dataset Exploration
- [ ] Section 3 — Feature and Target Identification
- [ ] Section 4 — Data Preparation (encoding justified)
- [ ] Section 5 — Train/Test Split
- [ ] Section 6 — Linear Regression Model
- [ ] Section 7 — Predictions (actual vs. predicted table)
- [ ] Section 8 — Coefficients and Intercept (full table + interpretation)
- [ ] Section 9 — Residual Analysis (plots + discussion)
- [ ] Section 10 — Regression Metrics (MAE, MSE, RMSE, R²)
- [ ] Section 11 — Training vs Testing Performance (comparison table + discussion)
- [ ] Section 12 — Model Interpretation (most influential features)
- [ ] Section 13 — Conclusion, submission.csv generated, and submitted to the Kaggle competition (score/rank recorded)
- [ ] Notebook exported/shared as a single Colab (.ipynb) file — no other files submitted